# Week 56, Day 2: Softmax regression for many classes

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w056/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Softmax regression from scratch

Write a numerically stable softmax (subtract the row maximum), and loss_grad(w_flat), returning the mean cross-entropy plus a small L2 penalty and its gradient, for use with scipy's L-BFGS. Fit and report test accuracy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
r = np.random.default_rng(562)
K, N = 8, 80
dirs = np.arange(K) * 2 * np.pi / K
pref = r.uniform(0, 2 * np.pi, N); depth = r.uniform(0.2, 1.0, N)

def trials(n, g):
    lab = g.integers(0, K, n)
    rates = 10 * (1 + depth * np.cos(dirs[lab][:, None] - pref))
    return g.poisson(rates * 0.2).astype(float), lab

X_tr, lab_tr = trials(320, r); X_te, lab_te = trials(2000, r)
mu, sd = X_tr.mean(0), X_tr.std(0) + 1e-9
A_tr, A_te = np.c_[np.ones(320), (X_tr - mu) / sd], np.c_[np.ones(2000), (X_te - mu) / sd]
Y_tr = np.eye(K)[lab_tr]

def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    return ...

def loss_grad(w_flat, lam=1e-2):
    W = w_flat.reshape(A_tr.shape[1], K)
    P = softmax(A_tr @ W)
    loss = -np.mean(np.log(P[np.arange(len(lab_tr)), lab_tr])) + 0.5 * lam * np.sum(W[1:] ** 2)
    G = ...
    G[1:] += lam * W[1:]
    return loss, G.ravel()

res = minimize(loss_grad, np.zeros(A_tr.shape[1] * K), jac=True, method="L-BFGS-B")
W = res.x.reshape(A_tr.shape[1], K)
P_te = softmax(A_te @ W)
acc_soft = np.mean(P_te.argmax(1) == lab_te)
print(f"converged: {res.success}; test accuracy {acc_soft:.3f} (chance {1 / K:.3f})")

In [ ]:
import numpy as np
def test_softmax():
    "softmax is stable for large scores and its rows sum to 1"
    P = softmax(np.array([[1000.0, 1001.0], [0.0, 0.0]]))
    assert np.allclose(P.sum(1), 1) and np.allclose(P[1], [0.5, 0.5]) and np.isfinite(P).all()
def test_grad():
    "loss_grad returns the loss and X'(P - Y) / n plus the penalty"
    W0 = 0.01 * np.ones((A_tr.shape[1], K)); L, G = loss_grad(W0.ravel())
    e = np.zeros_like(W0); e[3, 2] = 1e-6
    L2, _ = loss_grad((W0 + e).ravel()); assert abs((L2 - L) / 1e-6 - G.reshape(W0.shape)[3, 2]) < 1e-4
def test_acc():
    "test accuracy is between 0.6 and 0.95"
    assert 0.6 < acc_soft < 0.95

test_softmax()
test_grad()
test_acc()
print("All checks passed")

### Exercise: Eight binary classifiers

For each class k, fit a penalised binary logistic regression of (lab_tr == k) with L-BFGS, and store its test probabilities in column k of S. Compare accuracy with softmax and check how far the rows of S are from summing to one.

In [ ]:
def fit_binary(A, t, lam=1e-2):
    def f(w):
        p = 1 / (1 + np.exp(-A @ w)); p = np.clip(p, 1e-12, 1 - 1e-12)
        L = -np.mean(t * np.log(p) + (1 - t) * np.log(1 - p)) + 0.5 * lam * np.sum(w[1:] ** 2)
        G = A.T @ (p - t) / len(t); G[1:] += lam * w[1:]
        return L, G
    return minimize(f, np.zeros(A.shape[1]), jac=True, method="L-BFGS-B").x

S = np.zeros((2000, K))
for k in range(K):
    w_k = ...
    S[:, k] = ...
acc_ovr = np.mean(S.argmax(1) == lab_te)
print(f"one-vs-rest accuracy {acc_ovr:.3f}, softmax {acc_soft:.3f}")
print(f"one-vs-rest probability sums: mean {S.sum(1).mean():.2f}, range {S.sum(1).min():.2f} to {S.sum(1).max():.2f}")

In [ ]:
import numpy as np
def test_ovr():
    "S holds one column of probabilities per binary classifier"
    assert S.shape == (2000, K) and np.all((S > 0) & (S < 1))
def test_compare():
    "softmax is at least as accurate as one-vs-rest, whose probabilities do not sum to one"
    assert acc_soft >= acc_ovr and np.abs(S.sum(1) - 1).mean() > 0.05

test_ovr()
test_compare()
print("All checks passed")

### Exercise: Where the errors go

Build the row-normalised confusion matrix C. Compute the fraction of errors that land on a neighbouring direction (one step either way round the circle). Then, for 2,000 reaches in arbitrary directions (not only the eight trained ones), decode a continuous angle from the class probabilities Pc and compare its mean absolute angular error with that of the argmax decision.

In [ ]:
pred = P_te.argmax(1)
C = np.zeros((K, K))
for t_, p_ in zip(lab_te, pred):
    C[t_, p_] += 1
C = C / C.sum(1, keepdims=True)
wrong = pred != lab_te
step = (pred - lab_te) % K
frac_neighbour = ...
g3 = np.random.default_rng(5621)                     # reaches in any direction, not only the eight trained ones
theta = g3.uniform(0, 2 * np.pi, 2000)
Xc = g3.poisson(10 * (1 + depth * np.cos(theta[:, None] - pref)) * 0.2).astype(float)
Pc = softmax(np.c_[np.ones(2000), (Xc - mu) / sd] @ W)
ang_err = lambda est: np.abs(np.angle(np.exp(1j * (est - theta))))
theta_cont = ...
err_argmax, err_cont = np.degrees(ang_err(dirs[Pc.argmax(1)]).mean()), np.degrees(ang_err(theta_cont).mean())
print(f"errors on neighbouring directions: {frac_neighbour:.2f} of all errors")
print(f"mean angular error: argmax {err_argmax:.1f} degrees, probability-weighted {err_cont:.1f} degrees")
plt.imshow(C, cmap="viridis"); plt.xlabel("decoded direction"); plt.ylabel("true direction"); plt.colorbar(); plt.show()

In [ ]:
import numpy as np
def test_conf():
    "C is the 8 by 8 confusion matrix with rows normalised to 1"
    assert C.shape == (K, K) and np.allclose(C.sum(1), 1)
def test_neigh():
    "most errors go to the two neighbouring directions"
    assert frac_neighbour > 0.6
def test_angle():
    "for reaches in arbitrary directions, the continuous estimate has a smaller mean absolute error than the argmax estimate"
    assert err_cont < err_argmax

test_conf()
test_neigh()
test_angle()
print("All checks passed")